In [1]:
import pandas as pd
import numpy as np
import os
import requests
from datetime import datetime
from pathlib import Path

pd.set_option('display.max_columns', None)

In [2]:
# ── 1. PATH CONFIGURATION ───────────────
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parents[2]

BASE_DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR    = BASE_DATA_DIR / "raw"
BRONZE_DIR = BASE_DATA_DIR / "bronze"
SILVER_DIR = BASE_DATA_DIR / "silver"

RAW_DIR.mkdir(parents=True, exist_ok=True)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://www.data.gouv.fr/api/1/datasets/r/8fdb0926-ea9d-4fb4-a136-7767cd97e30b"

path_data_raw     = RAW_DIR / "PR17_BVot_T1_FE.txt"
path_rhone_bronze = BRONZE_DIR / "2017-pres-t1-commune-rhone-69-bronze.csv"
path_rhone_silver = SILVER_DIR / "2017-pres-t1-commune-rhone-69-silver.csv"
print(f"Raw data path: {path_data_raw}")
print(f"Bronze path: {path_rhone_bronze}")
print(f"Silver path: {path_rhone_silver}")


Raw data path: /Users/zainfrayha/Desktop/electio-analytics-poc/data/raw/PR17_BVot_T1_FE.txt
Bronze path: /Users/zainfrayha/Desktop/electio-analytics-poc/data/bronze/2017-pres-t1-commune-rhone-69-bronze.csv
Silver path: /Users/zainfrayha/Desktop/electio-analytics-poc/data/silver/2017-pres-t1-commune-rhone-69-silver.csv


In [3]:
# =============================================================================
# ── 2. RAW LAYER: Landing Zone 
# =============================================================================
if not os.path.exists(path_data_raw):
    print(f"📥 Downloading source to RAW...")
    resp = requests.get(DATA_URL, timeout=180)
    resp.raise_for_status()
    with open(path_data_raw, "wb") as f:
        f.write(resp.content)
    print(f"✅ Saved to RAW: {path_data_raw}")
else:
    print(f"✅ Raw file already exists in {RAW_DIR}")


✅ Raw file already exists in /Users/zainfrayha/Desktop/electio-analytics-poc/data/raw


In [4]:
# =============================================================================
# ── 3. BRONZE LAYER: Ingestion & Metadata (Bureau de Vote level)
# =============================================================================
print("\n🛠 Processing RAW -> BRONZE...")

COMMON_FIELDS = [
    "Code du département", "Libellé du département",
    "Code de la circonscription", "Libellé de la circonscription",
    "Code de la commune", "Libellé de la commune",
    "Code du b.vote", "Inscrits", "Abstentions", "% Abs/Ins",
    "Votants", "% Vot/Ins", "Blancs", "% Blancs/Ins", "% Blancs/Vot",
    "Nuls", "% Nuls/Ins", "% Nuls/Vot", "Exprimés", "% Exp/Ins", "% Exp/Vot",
]
CAND_FIELDS = ["N°Panneau", "Sexe", "Nom", "Prénom", "Voix", "% Voix/Ins", "% Voix/Exp"]
N_CANDIDATES = 11

# Build the 101 expected columns
all_columns = COMMON_FIELDS.copy()
for k in range(1, N_CANDIDATES + 1):
    for field in CAND_FIELDS:
        all_columns.append(f"{field}_{k}")

# Read ignoring the broken header
df_all = pd.read_csv(
    path_data_raw,
    sep=";",
    dtype=str,
    encoding="cp1252",
    header=None,
    skiprows=1,
    names=all_columns,
    engine="python"
)

# Clean basic strings & filter Rhône
df_all.columns = df_all.columns.str.strip()
df_all["Code du département"] = df_all["Code du département"].astype(str).str.strip().str.zfill(2)
df_bronze = df_all[df_all["Code du département"] == "69"].copy()

# Add metadata
df_bronze["extraction_source_url"] = DATA_URL
df_bronze["ingestion_timestamp"] = datetime.now().isoformat()
df_bronze["source_file_name"] = os.path.basename(path_data_raw)

df_bronze.to_csv(path_rhone_bronze, index=False, sep=";", encoding="utf-8")
print(f"✅ BRONZE dataset created: {path_rhone_bronze} ({len(df_bronze)} rows)")



🛠 Processing RAW -> BRONZE...


✅ BRONZE dataset created: /Users/zainfrayha/Desktop/electio-analytics-poc/data/bronze/2017-pres-t1-commune-rhone-69-bronze.csv (1287 rows)
